# 프로젝트 4 - Weekend 3: Multimodal Graph-RAG (Week 12 적용)

**목표** — Week 12에서 배운 **GraphRAG 패턴**을 Weekend 2의 **멀티모달 Document**에 적용합니다. PDF 표/이미지, 오디오, 비디오에서 *함께* 엔티티/관계를 추출해 단일 KG를 만들고, Leiden 커뮤니티 + 요약 + Local/Global Search까지 운영급으로 완성합니다.

**학습 목표**:
1. 멀티모달 Document → 청크별 LLM 엔티티/관계 추출 (`element_type` 메타 보존)
2. 청크 결과 병합 → `nx.DiGraph` (노드에 `source_modalities` 속성)
3. **Leiden 커뮤니티** 탐지 + modality 분포 분석
4. **LLM 커뮤니티 요약** (modality 정보 포함)
5. **Multimodal Local/Global Search** — 엔티티/요약 기반 + modality filter
6. **Multimodal Graph-RAG** — Week 12 패턴을 적용한 답변 생성
7. modality-aware 가중치 (표=정량, audio=발화, video=시각 등)
8. LLM-as-Judge로 Graph-RAG vs baseline 비교
9. `MultimodalGraphRAGService` 통합 클래스


In [1]:
# 환경 설정 및 라이브러리 설치
!pip install -q langchain langchain-openai langchain-community faiss-cpu \
    pymupdf pillow pandas pydantic networkx graspologic matplotlib python-dotenv


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
environs 15.0.1 requires marshmallow>=4.0.0, but you have marshmallow 3.26.2 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import io
import json
import base64
from pathlib import Path
from collections import Counter, defaultdict
from dotenv import load_dotenv

load_dotenv()

import fitz
import pandas as pd
import networkx as nx
from PIL import Image
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
vision = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

DATA_DIR = Path("data")
PDF_REPORT = DATA_DIR / "quarterly_report.pdf"
IMG_DIR = DATA_DIR / "_extracted"
IMG_DIR.mkdir(exist_ok=True)
assert PDF_REPORT.exists(), f"❌ {PDF_REPORT} 없음"
print("✅ 환경 설정 완료")


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


✅ 환경 설정 완료


---
## 📦 실습 데이터 + Weekend 2 사전 빌드

이 노트북은 Weekend 2의 결과물을 입력으로 가정합니다. Weekend 2를 끝내지 않았어도 아래 셀을 실행하면 자동으로 빌드됩니다:

| 객체 | 설명 |
|------|------|
| `mm_docs` | PDF 표/이미지/페이지 OCR → 멀티모달 Document 리스트 |
| `vs` | FAISS 인덱스 (`OpenAIEmbeddings`) |
| `weighted_search(...)` | `element_type` 가중치 검색 |
| `multimodal_rag_answer(...)` | 멀티모달 RAG 답변 (Graph-RAG의 baseline) |

> Weekend 2 노트북에서 더 큰 인덱스(오디오 + 비디오 포함)를 만들었다면 해당 인덱스를 사용해도 됩니다.


In [3]:
# === Weekend 2 결과 재현: build_multimodal_documents ===
def _b64_image_message(path, prompt):
    raw = Path(path).read_bytes()
    b64 = base64.b64encode(raw).decode()
    ext = Path(path).suffix.lstrip(".").lower()
    mime = "image/jpeg" if ext in ("jpg", "jpeg") else f"image/{ext}"
    return HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
    ])


def _extract_tables_md(pdf_path, src):
    doc = fitz.open(pdf_path)
    out = []
    for pi, page in enumerate(doc, start=1):
        try:
            tabs = page.find_tables()
        except Exception:
            continue
        for tb in tabs.tables:
            try:
                df = tb.to_pandas().dropna(how="all").dropna(axis=1, how="all")
            except Exception:
                continue
            if df.shape[0] < 1 or df.shape[1] < 2:
                continue
            try:
                md_text = df.to_markdown(index=False)
            except Exception:
                md_text = "\n".join(["| " + " | ".join(str(v) for v in row) + " |" for row in df.values.tolist()])
            out.append(Document(page_content=f"**Table p{pi}**\n\n{md_text}",
                                metadata={"source": src, "page_number": pi, "element_type": "table"}))
    doc.close()
    return out


def _extract_image_captions(pdf_path, img_dir, src, max_images=2):
    doc = fitz.open(pdf_path)
    docs = []
    saved = 0
    for pi, page in enumerate(doc, start=1):
        for ii, info in enumerate(page.get_images(full=True), start=1):
            if saved >= max_images:
                break
            data = doc.extract_image(info[0])
            ext = data["ext"]
            p = Path(img_dir) / f"{Path(pdf_path).stem}_p{pi}_img{ii}.{ext}"
            p.write_bytes(data["image"])
            cap = vision.invoke([_b64_image_message(str(p), "이 이미지를 한국어로 1~2문장으로 설명하세요.")]).content
            docs.append(Document(page_content=cap,
                                 metadata={"source": src, "page_number": pi, "element_type": "image_caption"}))
            saved += 1
        if saved >= max_images:
            break
    doc.close()
    return docs


def _page_ocr(pdf_path, img_dir, src, max_pages=1):
    docs = []
    doc = fitz.open(pdf_path)
    stem = Path(pdf_path).stem
    for pi, page in enumerate(doc, start=1):
        if pi > max_pages:
            break
        pix = page.get_pixmap(dpi=120)
        p = Path(img_dir) / f"{stem}_p{pi}.png"
        pix.save(str(p))
        ocr = vision.invoke([_b64_image_message(str(p),
            "이 이미지의 모든 텍스트를 원본 순서대로 추출하세요. 표는 행/열을 보존하세요. 해석 금지.")]).content
        docs.append(Document(page_content=ocr,
                             metadata={"source": src, "page_number": pi, "element_type": "page_ocr"}))
    doc.close()
    return docs


def build_multimodal_documents(pdf_path, img_dir):
    src = Path(pdf_path).name
    return (_extract_tables_md(pdf_path, src)
            + _extract_image_captions(pdf_path, img_dir, src, max_images=2)
            + _page_ocr(pdf_path, img_dir, src, max_pages=1))


mm_docs = build_multimodal_documents(str(PDF_REPORT), str(IMG_DIR))
print(f"멀티모달 Documents: {len(mm_docs)}")
print(Counter(d.metadata["element_type"] for d in mm_docs))


# === Weekend 2 결과 재현: FAISS 인덱스 + 가중치 검색 + 멀티모달 RAG ===
vs = FAISS.from_documents(mm_docs, embeddings)
print(f"\n✅ FAISS 인덱스 — {vs.index.ntotal} 벡터")


DEFAULT_WEIGHTS = {
    "table": 1.4, "page_ocr": 1.0, "image_caption": 0.8,
    "audio_chunk": 1.1, "video_frame_caption": 0.9, "video_audio_chunk": 1.1,
}


def weighted_search(vs, query, weights=None, k=5, fetch_k=20):
    w = DEFAULT_WEIGHTS if weights is None else weights
    hits = vs.similarity_search_with_score(query, k=fetch_k)
    scored = []
    for doc, distance in hits:
        et = doc.metadata.get("element_type", "")
        weight = w.get(et, 1.0)
        if weight == 0:
            continue
        scored.append((doc, weight / (distance + 1e-9)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


ELEMENT_LABELS = {
    "table": "표 (PDF)", "page_ocr": "페이지 텍스트", "image_caption": "이미지 캡션",
    "audio_chunk": "회의 발화", "video_frame_caption": "비디오 프레임", "video_audio_chunk": "비디오 발화",
}


def multimodal_rag_answer(vs, query, k=6):
    hits = weighted_search(vs, query, k=k)
    groups, sources = {}, []
    for doc, _ in hits:
        et = doc.metadata.get("element_type", "etc")
        groups.setdefault(et, []).append(doc.page_content)
        sources.append({"element_type": et, "snippet": doc.page_content[:80]})
    sections = []
    for et, contents in groups.items():
        body = "\n".join(f"- {c}" for c in contents)
        sections.append(f"### {ELEMENT_LABELS.get(et, et)}\n{body}")
    ctx = "\n\n".join(sections)
    sys = SystemMessage(content=(
        "당신은 멀티모달 RAG 어시스턴트. 컨텍스트의 각 섹션은 다른 모달리티에서 추출됨. "
        "근거 부족 시 '제공된 자료에서 확인 불가'라고 답하세요."
    ))
    user = HumanMessage(content=f"질문: {query}\n\n컨텍스트:\n{ctx}")
    return {"answer": llm.invoke([sys, user]).content, "sources": sources}


print("✅ weighted_search / multimodal_rag_answer 준비됨 (Graph-RAG의 baseline)")


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
멀티모달 Documents: 5
Counter({'table': 2, 'image_caption': 2, 'page_ocr': 1})

✅ FAISS 인덱스 — 5 벡터
✅ weighted_search / multimodal_rag_answer 준비됨 (Graph-RAG의 baseline)


---
## 문제 1: 멀티모달 Document에서 청크별 엔티티/관계 추출

`mm_docs`의 각 Document를 LLM에 보내 엔티티/관계를 JSON으로 추출. **modality 정보를 LLM에 알려주고**, 추출 결과 dict에 `element_type`을 보존합니다.

**요구사항:**
- 함수 시그니처: `extract_one_doc(doc: Document) -> dict`
- 프롬프트에 `element_type` 안내 ("이 텍스트는 표/이미지 캡션/오디오/비디오 ...에서 추출됨")
- 반환: `{"entities": [...], "relations": [...], "element_type": str, "source": str}`
- 파싱 실패 시 빈 entities/relations 반환

**평가기준:**
- 모든 mm_docs에 대해 호출 성공 (예외 없음)
- 첫 표(table) Document에서 적어도 1개 엔티티 추출
- 반환 dict에 4개 키 모두 존재


In [4]:
# ✅ 문제 1 정답
import re

EXTRACT_PROMPT_TMPL = """다음 텍스트에서 엔티티와 관계를 JSON으로 추출하세요.
출처: {element_type}
스키마:
{{"entities": [{{"name": '...', "type": "Person|Organisation|Product|Concept|Metric"}}],
 "relations": [{{"source": '...', "target": '...', "label": '...'}}]}}
JSON만 출력하세요. 다른 텍스트 금지."""


def strip_fence(t):
    t = t.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*", "", t); t = re.sub(r"\s*```$", "", t)
    return t


def extract_one_doc(doc: Document) -> dict:
    et = doc.metadata.get('element_type', 'unknown')
    src = doc.metadata.get('source', '')
    prompt = EXTRACT_PROMPT_TMPL.format(element_type=et)
    msgs = [SystemMessage(content=prompt),
            HumanMessage(content=doc.page_content[:3000])]
    raw = llm.invoke(msgs).content
    try:
        data = json.loads(strip_fence(raw))
        return {"entities": data.get("entities", []),
                "relations": data.get("relations", []),
                "element_type": et, "source": src}
    except (json.JSONDecodeError, AttributeError):
        return {"entities": [], "relations": [], "element_type": et, "source": src}


# 전체 추출 (시간 소요)
print(f"📦 {len(mm_docs)}개 Document 추출 중...")
extraction_results = []
for i, doc in enumerate(mm_docs):
    r = extract_one_doc(doc)
    extraction_results.append({"doc_index": i, **r})
    if i < 5:
        print(f"  [{i:2d}] {r['element_type']:<22} ents={len(r['entities'])}, rels={len(r['relations'])}")
total_e = sum(len(r['entities']) for r in extraction_results)
total_r = sum(len(r['relations']) for r in extraction_results)
print(f"\n...총 엔티티 {total_e}개 (중복 포함), 관계 {total_r}개")


📦 5개 Document 추출 중...
  [ 0] table                  ents=8, rels=15
  [ 1] table                  ents=6, rels=12
  [ 2] image_caption          ents=6, rels=3
  [ 3] image_caption          ents=3, rels=3
  [ 4] page_ocr               ents=6, rels=5

...총 엔티티 29개 (중복 포함), 관계 38개


---
## 문제 2: 청크 결과 병합 → 멀티모달 KG

청크별로 따로 추출한 결과를 하나의 `nx.DiGraph`로 병합. **modality 정보를 노드/엣지 속성으로 보존**해 나중에 Local Search에서 modality filter가 가능하게.

**요구사항:**
- 함수 시그니처: `merge_multimodal_graph(extraction_results: list[dict]) -> nx.DiGraph`
- 노드 속성: `type`, `source_doc_indices` (list[int]), `source_modalities` (list[str], 중복 X)
- 엣지 속성: `relations` (list[str], 중복 X), `source_modalities` (list[str], 중복 X)

**평가기준:**
- 여러 청크에 등장하는 엔티티가 *하나의 노드*로 병합
- 적어도 한 노드의 `source_modalities` 길이 >= 2 (여러 modality에 등장하는 엔티티)


In [5]:
# ✅ 문제 2 정답
def merge_multimodal_graph(extraction_results: list[dict]) -> nx.DiGraph:
    G = nx.DiGraph()
    for r in extraction_results:
        di = r['doc_index']
        et = r['element_type']
        for e in r['entities']:
            name = e['name']
            if name in G:
                G.nodes[name]['source_doc_indices'].append(di)
                if et not in G.nodes[name]['source_modalities']:
                    G.nodes[name]['source_modalities'].append(et)
            else:
                G.add_node(name, type=e.get('type', 'Unknown'),
                           source_doc_indices=[di], source_modalities=[et])
        for rel in r['relations']:
            s, t = rel['source'], rel['target']
            label = rel.get('label', '')
            for ep in (s, t):
                if ep not in G:
                    G.add_node(ep, type="Unknown",
                               source_doc_indices=[di], source_modalities=[et])
            if G.has_edge(s, t):
                if label and label not in G[s][t]['relations']:
                    G[s][t]['relations'].append(label)
                if et not in G[s][t]['source_modalities']:
                    G[s][t]['source_modalities'].append(et)
            else:
                G.add_edge(s, t,
                           relations=[label] if label else [],
                           source_modalities=[et])
    return G


G = merge_multimodal_graph(extraction_results)
print(f"멀티모달 KG — 노드 {G.number_of_nodes()}, 엣지 {G.number_of_edges()}")
multi_mod = [(n, d) for n, d in G.nodes(data=True) if len(d.get("source_modalities", [])) >= 2]
print(f"\n여러 모달리티에 등장하는 엔티티 {len(multi_mod)}개 (상위 5개):")
for n, d in sorted(multi_mod, key=lambda x: -len(x[1]['source_modalities']))[:5]:
    print(f"  • {n} ({d.get('type', '?')}; in {d['source_modalities']})")


멀티모달 KG — 노드 26, 엣지 38

여러 모달리티에 등장하는 엔티티 5개 (상위 5개):
  • AI 솔루션 (Product; in ['table', 'page_ocr'])
  • 클라우드 (Product; in ['table', 'page_ocr'])
  • 컨설팅 (Product; in ['table', 'page_ocr'])
  • 기타 (Product; in ['table', 'page_ocr'])
  • 매출 (Metric; in ['table', 'image_caption'])


---
## 문제 3: Leiden 커뮤니티 탐지 + modality 분포

Leiden으로 KG를 *주제 그룹*으로 분할. 각 커뮤니티의 **modality 분포**를 같이 계산해 "이 커뮤니티는 어느 모달리티 정보 위주인가" 진단.

**요구사항:**
- 함수 시그니처: `detect_communities(G: nx.DiGraph) -> dict[int, dict]`
- 반환: `{cid: {"nodes": [name], "modality_counts": {mod: count}, "size": int}}`
- `graspologic.partition.leiden(G.to_undirected())` 사용

**평가기준:**
- 커뮤니티 1개 이상
- 각 커뮤니티 dict에 `nodes`, `modality_counts`, `size` 키 존재


In [6]:
# ✅ 문제 3 정답
from graspologic.partition import leiden


def detect_communities(G: nx.DiGraph) -> dict:
    parts = leiden(G.to_undirected())
    out = {}
    for node, cid in parts.items():
        out.setdefault(cid, {"nodes": [], "modality_counts": Counter()})
        out[cid]['nodes'].append(node)
        for mod in G.nodes[node].get("source_modalities", []):
            out[cid]['modality_counts'][mod] += 1
    # size 추가 + Counter → dict
    for cid, info in out.items():
        info['size'] = len(info['nodes'])
        info['modality_counts'] = dict(info['modality_counts'])
    return out


communities = detect_communities(G)
print(f"커뮤니티 {len(communities)}개:")
for cid in sorted(communities):
    info = communities[cid]
    mods = ", ".join(f"{m}:{c}" for m, c in sorted(info['modality_counts'].items(), key=lambda x: -x[1]))
    print(f"  C{cid} ({info['size']}노드)  [{mods}]")
    print(f"    nodes: {info['nodes'][:5]}{'...' if info['size']>5 else ''}")


커뮤니티 4개:
  C0 (9노드)  [table:7, page_ocr:6]
    nodes: ['Modu Tech', '기타', '2025년 4분기 실적 보고서', '영업이익', '컨설팅']...
  C1 (3노드)  [image_caption:3, table:1]
    nodes: ['시장 평균', '매출', '시간']
  C2 (7노드)  [table:7]
    nodes: ['MAU', 'NPS', '2025 Q4', '2024 Q4', '이탈률']...
  C3 (4노드)  [image_caption:4]
    nodes: ['Q4', 'Q3', 'Q1', 'Q2']


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/graspologic/partition/leiden.py:338: UserWarning: Leiden partitions do not contain all nodes from the input graph because input graph contained isolate nodes.
  warnings.warn(


---
## 문제 4: LLM 커뮤니티 요약 (modality 정보 포함)

각 커뮤니티에 대해 LLM에게 *엔티티 + 관계 + modality 분포*를 주고 한 단락 요약을 받습니다. modality 정보를 포함해야 LLM이 "이 커뮤니티는 표/회의 발화에서 주로 나온 거" 같은 *증거 출처*를 답에 반영.

**요구사항:**
- 함수 시그니처: `summarize_multimodal_community(G: nx.DiGraph, community_info: dict) -> str`
- 프롬프트에 modality_counts, 엔티티 목록(이름 + 출처 모달리티), 내부 관계 포함
- 4-6 문장 한국어 요약

**평가기준:**
- 빈 문자열 아님
- 가능하면 modality 단어("표", "회의", "비디오" 등)가 1회 이상 등장 (LLM 의존)


In [7]:
# ✅ 문제 4 정답
COMMUNITY_PROMPT = """다음은 멀티모달 지식 그래프의 한 커뮤니티입니다.
모달리티 분포(어느 데이터 출처에서 왔는지): {modality_counts}

엔티티:
{entities}

내부 관계:
{relations}

이 커뮤니티의 *주제*와 *주요 사실*을 4-6 문장의 한국어로 요약하세요.
가능하면 어느 모달리티에서 나온 정보인지 짧게 언급해 주세요."""


def summarize_multimodal_community(G: nx.DiGraph, community_info: dict) -> str:
    nodes = community_info['nodes']
    nodes_set = set(nodes)
    ent_lines = []
    for n in nodes:
        d = G.nodes[n]
        mods = ", ".join(d.get("source_modalities", []))
        ent_lines.append(f"- {n} ({d.get('type', '?')}; from: {mods})")
    rel_lines = []
    for u, v, ed in G.edges(data=True):
        if u in nodes_set and v in nodes_set:
            for rel in (ed.get('relations') or [""]):
                rel_lines.append(f"- {u} --[{rel}]--> {v}")
    prompt = COMMUNITY_PROMPT.format(
        modality_counts=community_info['modality_counts'],
        entities="\n".join(ent_lines) or "(없음)",
        relations="\n".join(rel_lines) or "(없음)",
    )
    return llm.invoke([HumanMessage(content=prompt)]).content


# 모든 커뮤니티 요약 (시간 소요)
community_summaries = {}
for cid in sorted(communities):
    print(f"  요약 중: C{cid}...")
    community_summaries[cid] = summarize_multimodal_community(G, communities[cid])
print(f"\n=== 첫 요약 미리보기 ===")
first_cid = sorted(community_summaries)[0]
print(f"[C{first_cid}] {community_summaries[first_cid][:300]}...")


  요약 중: C0...
  요약 중: C1...
  요약 중: C2...
  요약 중: C3...

=== 첫 요약 미리보기 ===
[C0] 이 커뮤니티는 Modu Tech이라는 조직이 2025년 4분기 실적 보고서를 발행하고, AI 솔루션, 클라우드, 컨설팅, 기타 제품을 사업부로 운영하고 있다는 내용을 담고 있습니다. 이와 관련하여 영업이익과 전년동기 대비 지표가 여러 제품과 연결되어 있으며, 이는 주로 페이지 OCR과 테이블에서 수집된 정보입니다. Modu Tech의 다양한 사업부는 각각 영업이익에 기여하고 있으며, 이들 간의 관계가 명확히 정리되어 있습니다. 이러한 정보는 Modu Tech의 성과와 사업 전략을 이해하는 데 중요한 역할을 합니다....


---
## 문제 5: Multimodal Local Search — modality filter 지원

질문에서 엔티티를 매칭 → 이웃 → 원본 mm_docs 컨텍스트로 답변. **modality_filter** 옵션으로 "표만 보고 답하라" 같은 제약 가능.

**요구사항:**
- 함수 시그니처:
  ```python
  multimodal_local_search(question: str, G: nx.DiGraph, mm_docs: list[Document],
                          hops: int = 1, modality_filter: list[str] | None = None) -> str
  ```
- 동작:
  1. G의 노드 중 이름이 question에 포함된 것 찾기
  2. 매칭 노드의 `hops` 홉 이웃 확장
  3. 이웃 노드의 `source_doc_indices` 모으기
  4. `modality_filter`가 주어지면 `mm_docs[i].metadata['element_type']`이 그 안에 있는 것만
  5. 청크 텍스트 + 내부 엣지 정보를 컨텍스트로 LLM 호출

**평가기준:**
- 매칭 엔티티 없으면 "(매칭 없음)" 같은 안내 반환
- `modality_filter=['table']`로 호출 시 컨텍스트에 audio_chunk가 안 들어감


In [8]:
# ✅ 문제 5 정답
def multimodal_local_search(question: str, G: nx.DiGraph,
                            mm_docs: list, hops: int = 1,
                            modality_filter: list | None = None) -> str:
    matched = [n for n in G.nodes if n in question]
    if not matched:
        return "(질문에서 매칭되는 엔티티 없음 — Global Search를 시도하세요)"
    nodes = set(matched)
    for _ in range(hops):
        new = set()
        for n in nodes:
            new |= set(G.successors(n)) | set(G.predecessors(n))
        nodes |= new
    # 청크 인덱스 + modality filter
    doc_ids = set()
    for n in nodes:
        for di in G.nodes[n].get("source_doc_indices", []):
            et = mm_docs[di].metadata.get('element_type', '')
            if modality_filter is None or et in modality_filter:
                doc_ids.add(di)
    if not doc_ids:
        return "(modality filter 적용 후 컨텍스트 없음)"
    chunk_text = "\n\n".join(
        f"[{mm_docs[i].metadata.get('element_type', '?')}] {mm_docs[i].page_content}"
        for i in sorted(doc_ids)
    )
    # 엣지 정보
    edge_lines = []
    for u, v, d in G.edges(data=True):
        if u in nodes and v in nodes:
            for rel in (d.get('relations') or [""]):
                edge_lines.append(f"({u}) --[{rel}]--> ({v})")
    edge_text = "\n".join(edge_lines)
    sys = SystemMessage(content=(
        "주어진 멀티모달 컨텍스트(각 청크는 [모달리티] 접두사)와 그래프 관계만으로 한국어로 답하세요. "
        "관계를 따라 *다중 홉 추론*도 가능합니다. 답 없으면 '문서에서 찾을 수 없음'."
    ))
    user = HumanMessage(content=f"질문: {question}\n\n관계:\n{edge_text}\n\n청크:\n{chunk_text}")
    return llm.invoke([sys, user]).content


# 표 + OCR만 사용
ans = multimodal_local_search(
    "Modu Tech의 Q4 매출은 얼마인가요?",
    G, mm_docs, hops=1, modality_filter=["table", "page_ocr"],
)
print(f"💬 (table+page_ocr only)\n{ans}")


💬 (table+page_ocr only)
Modu Tech의 Q4 매출은 188억원입니다.


---
## 문제 6: Multimodal Global Search — 커뮤니티 요약 종합

추상적/요약형 질문은 *모든 커뮤니티 요약*을 종합해 답.

**요구사항:**
- 함수 시그니처: `multimodal_global_search(question: str, community_summaries: dict, communities: dict) -> str`
- 프롬프트에 각 커뮤니티의 *modality 분포*도 같이 보여줌 → LLM이 "표 위주 커뮤니티 / 회의 위주 커뮤니티" 구별 가능
- 4-8 문장 한국어 답변

**평가기준:**
- 답변 길이 >= 50자


In [9]:
# ✅ 문제 6 정답
def multimodal_global_search(question: str, community_summaries: dict, communities: dict) -> str:
    sections = []
    for cid in sorted(community_summaries):
        mod = communities[cid]['modality_counts']
        sections.append(f"[C{cid} | modality: {mod}]\n{community_summaries[cid]}")
    summary_text = "\n\n".join(sections)
    sys = SystemMessage(content=(
        "당신은 멀티모달 지식 그래프의 *커뮤니티 요약*들을 종합해 답하는 어시스턴트입니다. "
        "각 커뮤니티는 모달리티 분포 표시가 있습니다. 4-8 문장 한국어로 답하세요."
    ))
    user = HumanMessage(content=f"질문: {question}\n\n=== 커뮤니티 요약 ===\n{summary_text}")
    return llm.invoke([sys, user]).content


print(multimodal_global_search(
    "이 문서/회의/영상에서 다루는 핵심 주제는 무엇이고, 모달리티별로 어떤 정보가 보강되었나요?",
    community_summaries, communities,
))


이 문서/회의/영상의 핵심 주제는 Modu Tech의 2025년 4분기 실적 보고서와 관련된 다양한 사업부의 성과 및 지표 분석입니다. 커뮤니티 C0에서는 페이지 OCR과 테이블을 통해 영업이익과 전년 대비 지표를 연결하여 Modu Tech의 사업 전략을 이해하는 데 중요한 정보를 제공합니다. C1에서는 매출과 시장 평균의 비교 및 시간에 따른 추세를 이미지 캡션과 표를 통해 분석하고 있습니다. C2는 2024년과 2025년 4분기 동안의 MAU, NPS, 이탈률 등 주요 지표를 테이블로 정리하여 기업의 성과와 고객 유지 전략을 평가하는 데 도움을 줍니다. 마지막으로 C3에서는 이미지 캡션을 통해 수익 증가의 연쇄적인 관계를 설명하며, 각 메트릭 간의 상관관계를 강조하고 있습니다. 이러한 다양한 모달리티의 정보는 Modu Tech의 성과 분석과 전략 수립에 중요한 역할을 합니다.


---
## 문제 7: Multimodal Graph-RAG — auto 라우팅 + baseline 결합

질문 → *엔티티 매칭 여부*로 Local/Global 자동 선택. Local 결과 + baseline FAISS 결과를 *함께* 사용해 더 풍부한 답변.

**요구사항:**
- 함수 시그니처:
  ```python
  multimodal_graphrag_answer(question: str, vs, G, mm_docs, community_summaries, communities,
                              hops: int = 1, modality_filter: list | None = None) -> dict
  ```
- 동작:
  1. G의 노드 중 question에 매칭되는 것 있나?
     - 있으면 Local + baseline FAISS top-3 결합
     - 없으면 Global
  2. 반환: `{"answer", "mode": "local"|"global", "matched_entities": [...]}`

**평가기준:**
- 엔티티가 있는 질문(예: "Modu Tech ...")은 mode="local"
- 추상적 질문(예: "전반적 트렌드")은 mode="global"


In [10]:
# ✅ 문제 7 정답
def multimodal_graphrag_answer(
    question: str, vs, G: nx.DiGraph, mm_docs: list,
    community_summaries: dict, communities: dict,
    hops: int = 1, modality_filter: list | None = None,
) -> dict:
    matched = [n for n in G.nodes if n in question]
    if matched:
        # Local + baseline FAISS 결합
        local_ans = multimodal_local_search(question, G, mm_docs, hops=hops,
                                            modality_filter=modality_filter)
        baseline_hits = vs.similarity_search(question, k=3)
        baseline_ctx = "\n".join(
            f"[{h.metadata.get('element_type', '?')}] {h.page_content[:300]}"
            for h in baseline_hits
        )
        sys = SystemMessage(content=(
            "당신은 멀티모달 Graph-RAG. Graph 답변과 baseline 검색 결과를 함께 보고 "
            "더 정확하고 풍부한 답변을 한국어로 작성하세요."
        ))
        user = HumanMessage(content=(
            f"질문: {question}\n\n[Graph 답변]\n{local_ans}\n\n[baseline 검색]\n{baseline_ctx}"
        ))
        final = llm.invoke([sys, user]).content
        return {"answer": final, "mode": "local", "matched_entities": matched}
    else:
        # Global
        ans = multimodal_global_search(question, community_summaries, communities)
        return {"answer": ans, "mode": "global", "matched_entities": []}


for q in [
    "Modu Tech의 Q4 매출과 회의에서 언급된 전략은?",
    "이 데이터의 전반적인 트렌드는?",
]:
    r = multimodal_graphrag_answer(q, vs, G, mm_docs, community_summaries, communities, hops=1)
    print(f"\n[{r['mode']}] 매칭: {r['matched_entities']}")
    print(f"  Q: {q}")
    print(f"  A: {r['answer'][:300]}...")



[local] 매칭: ['매출', 'Q4', 'Modu Tech']
  Q: Modu Tech의 Q4 매출과 회의에서 언급된 전략은?
  A: Modu Tech의 2025년 4분기 매출은 188억원으로, 전년 동기 대비 35% 성장한 수치입니다. 이 기간 동안 영업이익은 24억원을 기록했습니다. 특히, 실 솔루션 부문이 견조한 성장세를 보이며 전체 매출 증가에 크게 기여했습니다. 

회의에서 언급된 전략으로는 신규 엔터프라이즈 고객 12사를 확보하여 매출을 증가시키는 것이었습니다. 이러한 전략은 고객 수를 1,820명으로 늘리는 데 기여했으며, 이는 전년 대비 45.6% 증가한 수치입니다. 또한, 월간 활성 사용자(MAU) 수는 132만으로, 55.3% 증가했습니다. 고객...

[global] 매칭: []
  Q: 이 데이터의 전반적인 트렌드는?
  A: 이 데이터의 전반적인 트렌드는 Modu Tech의 사업 성과와 관련된 다양한 지표들이 서로 연결되어 있으며, 시간이 지남에 따라 긍정적인 변화를 보이고 있다는 것입니다. Modu Tech은 AI 솔루션, 클라우드, 컨설팅 등 여러 사업부를 운영하며, 각 사업부의 영업이익이 전체 성과에 기여하고 있습니다. 또한, 매출과 시장 평균의 비교를 통해 시장 내 위치를 분석하고 있으며, 고객 수, MAU, NPS, 이탈률 등의 지표를 통해 고객 유지 전략을 강화하고 있습니다. 마지막으로, 수익 증가가 연쇄적으로 이어지는 경향이 있어, 각 분기...


---
## 문제 8: query type에 따른 modality 가중치 자동 선택

질문 유형(정량/논의/시각)에 따라 modality 가중치가 다름. LLM으로 유형 분류 → 적합한 가중치 적용.

**요구사항:**
- Pydantic 모델: `class QueryType(BaseModel): type: Literal["quantitative", "discussion", "visual", "general"]`
- 함수 시그니처:
  ```python
  classify_and_answer(question: str, vs, G, mm_docs, community_summaries, communities) -> dict
  ```
- 동작:
  1. `llm.with_structured_output(QueryType)`로 분류
  2. type별 가중치 사전 사용:
     - quantitative → table/page_ocr 우대
     - discussion → audio_chunk/video_audio_chunk 우대
     - visual → image_caption/video_frame_caption 우대
     - general → 기본 가중치
  3. `multimodal_graphrag_answer(..., modality_filter=top_modalities)` 호출

**평가기준:**
- 반환 dict에 `query_type`, `answer`, `applied_modalities` 키


In [11]:
# ✅ 문제 8 정답
from typing import Literal


class QueryType(BaseModel):
    type: Literal["quantitative", "discussion", "visual", "general"] = Field(
        ..., description="질문 유형 분류")
    reasoning: str = Field(..., description="짧은 분류 근거(한국어)")


MODALITY_PROFILES = {
    "quantitative": ["table", "page_ocr"],
    "discussion":   ["audio_chunk", "video_audio_chunk", "page_ocr"],
    "visual":       ["image_caption", "video_frame_caption", "page_ocr"],
    "general":      None,
}


def classify_and_answer(question: str, vs, G, mm_docs,
                        community_summaries, communities) -> dict:
    classifier = llm.with_structured_output(QueryType)
    sys = SystemMessage(content=(
        "질문을 다음 4가지 중 하나로 분류: "
        "quantitative(숫자/수치/비율), discussion(발언/회의/의견), "
        "visual(이미지/차트/시각), general(그 외)."
    ))
    verdict = classifier.invoke([sys, HumanMessage(content=question)])
    mods = MODALITY_PROFILES.get(verdict.type)
    r = multimodal_graphrag_answer(
        question, vs, G, mm_docs, community_summaries, communities,
        hops=1, modality_filter=mods,
    )
    return {"query_type": verdict.type, "answer": r['answer'],
            "applied_modalities": mods, "classification_reason": verdict.reasoning}


for q in [
    "Q4 영업이익이 얼마인가요?",
    "회의에서 누가 어떤 발언을 했나요?",
    "비디오에서 보여준 그래프 모양은?",
]:
    r = classify_and_answer(q, vs, G, mm_docs, community_summaries, communities)
    print(f"\n[{r['query_type']}] {q}")
    print(f"  분류 근거: {r['classification_reason']}")
    print(f"  적용 modality: {r['applied_modalities']}")
    print(f"  답변: {r['answer'][:200]}...")


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryType(type='quantitat...터에 해당한다."), input_type=QueryType])
  return self.__pydantic_serializer__.to_python(



[quantitative] Q4 영업이익이 얼마인가요?
  분류 근거: 질문은 특정 수치인 '영업이익'에 대한 정보를 요청하고 있으므로, 이는 수치적 데이터에 해당한다.
  적용 modality: ['table', 'page_ocr']
  답변: 2025년 4분기 영업이익은 24억원입니다. 이는 전년 동기 대비 매출이 35% 성장한 결과로, AI 솔루션 부문이 특히 견조한 성장세를 보이며 전체 매출 증가에 기여했습니다....


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryType(type='discussio...련된 주제이다.'), input_type=QueryType])
  return self.__pydantic_serializer__.to_python(



[discussion] 회의에서 누가 어떤 발언을 했나요?
  분류 근거: 질문은 회의에서의 발언에 대한 내용을 묻고 있으므로, 발언이나 의견과 관련된 주제이다.
  적용 modality: ['audio_chunk', 'video_audio_chunk', 'page_ocr']
  답변: 회의에서 Modu Tech의 2025년 4분기 실적 보고서와 관련된 발언이 있었습니다. 이 보고서에서는 AI 솔루션, 클라우드, 컨설팅 등 다양한 사업부의 영업이익과 전년 동기 대비 지표가 논의되었습니다. 또한, 매출과 시장 평균 간의 비교 및 시간에 따른 추세 분석도 이루어졌습니다. 주요 지표인 MAU, NPS, 이탈률, 고객 수 등의 변화율에 대한 정보...


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryType(type='visual', ...소에 해당한다.'), input_type=QueryType])
  return self.__pydantic_serializer__.to_python(



[visual] 비디오에서 보여준 그래프 모양은?
  분류 근거: 질문이 그래프의 모양에 대한 것이므로 시각적 요소에 해당한다.
  적용 modality: ['image_caption', 'video_frame_caption', 'page_ocr']
  답변: 비디오에서 보여준 그래프는 Modu Tech의 2025년 4분기 실적 보고서와 관련된 다양한 지표들을 시각적으로 표현하고 있습니다. 이 그래프는 영업이익, 매출, 고객 수, MAU, NPS, 이탈률 등 여러 사업부의 성과를 비교하는 데 중점을 두고 있습니다. 특히, 각 지표 간의 관계와 변화율을 명확히 나타내어, 기업의 성과와 전략을 이해하는 데 도움을 줍...


---
## 문제 9: Graph-RAG vs baseline FAISS — LLM-as-Judge 비교

같은 질문에 대해 *baseline RAG*와 *Graph-RAG* 두 답변을 받아 LLM이 비교 평가. Graph-RAG가 실제로 더 나은지 정량 측정.

**요구사항:**
- Pydantic 모델: `class Comparison(BaseModel): better: Literal["baseline", "graphrag", "tie"]; reasoning: str`
- 함수 시그니처: `judge_comparison(question, vs, G, mm_docs, community_summaries, communities) -> dict`
- 두 답변 다 받고 → judge LLM 호출 → 어느 게 나은지 결정
- 반환: `{"baseline_answer", "graphrag_answer", "better", "reasoning"}`

**평가기준:**
- 반환 dict에 4개 키 모두 존재
- `better`는 3가지 값 중 하나


In [12]:
# ✅ 문제 9 정답
class Comparison(BaseModel):
    better: Literal["baseline", "graphrag", "tie"] = Field(...)
    reasoning: str = Field(..., description="짧은 비교 근거(한국어)")


def judge_comparison(question: str, vs, G, mm_docs,
                     community_summaries, communities) -> dict:
    baseline = multimodal_rag_answer(vs, question)['answer']
    graphrag = multimodal_graphrag_answer(
        question, vs, G, mm_docs, community_summaries, communities, hops=1,
    )['answer']
    judge_llm = llm.with_structured_output(Comparison)
    sys = SystemMessage(content=(
        "두 RAG 답변 중 어느 것이 더 충실하고 풍부한지 평가. "
        "Graph-RAG는 엔티티 관계/커뮤니티 정보를 더 쓸 수 있어 multi-hop이나 모달리티 횡단에 강할 수 있음."
    ))
    user = HumanMessage(content=(
        f"질문: {question}\n\n[A: baseline]\n{baseline}\n\n[B: graphrag]\n{graphrag}"
    ))
    verdict = judge_llm.invoke([sys, user])
    return {"baseline_answer": baseline, "graphrag_answer": graphrag,
            "better": verdict.better, "reasoning": verdict.reasoning}


q = "Modu Tech 매출 추이를 표와 회의 발언을 종합해 설명해주세요"
r = judge_comparison(q, vs, G, mm_docs, community_summaries, communities)
print(f"❓ {q}")
print(f"\n[baseline]\n{r['baseline_answer'][:300]}...")
print(f"\n[graphrag]\n{r['graphrag_answer'][:300]}...")
print(f"\n⚖️  Better: {r['better']}\n   Reason: {r['reasoning']}")


❓ Modu Tech 매출 추이를 표와 회의 발언을 종합해 설명해주세요

[baseline]
Modu Tech의 2025년 4분기 매출은 188억원으로, 전년 동기 대비 35% 성장했습니다. 이 성장은 주로 실 솔루션 부문에서의 견조한 성장에 기인하며, 신규 엔터프라이즈 고객 12사를 확보한 결과로 나타났습니다. 

분기별 매출 추이를 보면, 2025년 4분기 매출은 전 분기 대비 약 16% 증가했습니다. 사업부별로 살펴보면, AI 솔루션 부문이 98억원의 매출을 기록하며 42%의 성장을 보였고, 클라우드 부문은 54억원으로 18% 성장했습니다. 컨설팅 부문은 24억원으로 5% 성장했으며, 기타 부문은 12억원으로 2% 감...

[graphrag]
Modu Tech의 2025년 4분기 매출 추이는 매우 긍정적인 성과를 보여주고 있습니다. 매출은 188억원으로, 전년 동기 대비 35% 성장했습니다. 이 성장은 주로 AI 솔루션 부문에서의 42%라는 높은 성장률에 의해 주도되었습니다. 클라우드 부문도 18% 성장하며 긍정적인 실적을 기록했지만, 컨설팅 부문은 5%의 소폭 증가에 그쳤습니다. 반면, 기타 부문은 2% 감소하는 모습을 보였습니다.

아래의 표는 각 사업부의 매출과 영업이익, 그리고 전년 동기 대비 성장률을 정리한 것입니다:

| 사업부    | 매출(억) | 영업이익...

⚖️  Better: graphrag
   Reason: Graph-RAG는 매출 추이를 표로 정리하여 시각적으로 명확하게 제시하고, 각 사업부의 영업이익과 성장률을 포함하여 더 풍부한 정보를 제공합니다. 또한, 회의 발언을 인용하여 실적에 대한 긍정적인 평가를 강조함으로써, 매출 성장의 배경과 맥락을 잘 설명하고 있습니다. 반면, baseline은 데이터는 제공하지만, 시각적 요소나 회의 발언의 맥락을 포함하지 않아 상대적으로 덜 충실합니다.


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Comparison(better='graphr... 덜 충실합니다.'), input_type=Comparison])
  return self.__pydantic_serializer__.to_python(


---
## 문제 10: 최종 미션 — `MultimodalGraphRAGService` 통합 클래스 🏁

문제 1-9의 모든 단계를 단일 클래스로 캡슐화. 운영 시 한 객체로 인덱싱 + 쿼리 + 비교 모두.

**요구사항:**
- 클래스 시그니처:
  ```python
  class MultimodalGraphRAGService:
      def __init__(self, mm_docs: list[Document], vs: FAISS): ...
      def index(self) -> None: ...                                 # 1-4 단계
      def query(self, question: str, mode: str = "auto", modality_filter=None) -> dict: ...
      def stats(self) -> dict: ...                                 # 인덱싱 통계
      def compare(self, question: str) -> dict: ...                # 문제 9 재사용
  ```

- `mode`:
  - `"auto"` (기본) — 엔티티 매칭 → local, 없으면 global
  - `"local"` — 강제 local
  - `"global"` — 강제 global
  - `"adaptive"` — 문제 8의 classify_and_answer 사용

**평가기준:**
- `svc.index()` 호출 후 `svc.stats()` 결과에 노드/엣지/커뮤니티 수 포함
- `svc.query('...' mode="adaptive")` 동작
- `svc.compare('...')` 결과에 4개 키 (`baseline_answer`, `graphrag_answer`, `better`, `reasoning`)


In [13]:
# ✅ 문제 10 정답
class MultimodalGraphRAGService:
    """멀티모달 Graph-RAG 통합 서비스."""

    def __init__(self, mm_docs: list, vs: FAISS):
        self.mm_docs = mm_docs
        self.vs = vs
        self.extraction_results = None
        self.G = None
        self.communities = None
        self.summaries = None

    def index(self) -> None:
        print(f"📦 청크 추출 ({len(self.mm_docs)}개)...")
        self.extraction_results = []
        for i, d in enumerate(self.mm_docs):
            r = extract_one_doc(d)
            self.extraction_results.append({"doc_index": i, **r})
        print(f"🔗 그래프 병합...")
        self.G = merge_multimodal_graph(self.extraction_results)
        print(f"🎯 Leiden 커뮤니티 탐지...")
        self.communities = detect_communities(self.G)
        print(f"📝 커뮤니티 요약 ({len(self.communities)}개)...")
        self.summaries = {
            cid: summarize_multimodal_community(self.G, info)
            for cid, info in self.communities.items()
        }
        print(f"✅ 인덱싱 완료")

    def stats(self) -> dict:
        if self.G is None:
            return {"status": "not indexed"}
        modality_counts = Counter()
        for d in self.mm_docs:
            modality_counts[d.metadata.get('element_type', '?')] += 1
        return {
            "nodes": self.G.number_of_nodes(),
            "edges": self.G.number_of_edges(),
            "communities": len(self.communities),
            "modality_doc_counts": dict(modality_counts),
        }

    def query(self, question: str, mode: str = "auto",
              modality_filter: list | None = None) -> dict:
        if mode == "global":
            ans = multimodal_global_search(question, self.summaries, self.communities)
            return {"answer": ans, "mode": "global", "matched_entities": []}
        if mode == "local":
            ans = multimodal_local_search(question, self.G, self.mm_docs,
                                          hops=1, modality_filter=modality_filter)
            return {"answer": ans, "mode": "local", "matched_entities": []}
        if mode == "adaptive":
            return classify_and_answer(question, self.vs, self.G, self.mm_docs,
                                       self.summaries, self.communities)
        # auto
        return multimodal_graphrag_answer(
            question, self.vs, self.G, self.mm_docs,
            self.summaries, self.communities, hops=1, modality_filter=modality_filter,
        )

    def compare(self, question: str) -> dict:
        return judge_comparison(question, self.vs, self.G, self.mm_docs,
                                self.summaries, self.communities)


svc = MultimodalGraphRAGService(mm_docs, vs)
svc.index()
print("\n📊 stats:", svc.stats())

for q, mode in [
    ("Modu Tech Q4 매출은?", "auto"),
    ("회의에서 다룬 주요 결정은?", "adaptive"),
    ("코퍼스 전반 주제는?", "global"),
]:
    r = svc.query(q, mode=mode)
    print(f"\n[{r.get('mode', mode)}] {q}")
    print(f"  → {r.get('answer', '')[:200]}...")

cmp = svc.compare("매출 추이를 표와 회의 발언으로 설명")
print(f"\n⚖️  Comparison: {cmp['better']} — {cmp['reasoning']}")


📦 청크 추출 (5개)...
🔗 그래프 병합...
🎯 Leiden 커뮤니티 탐지...
📝 커뮤니티 요약 (5개)...
✅ 인덱싱 완료

📊 stats: {'nodes': 27, 'edges': 39, 'communities': 5, 'modality_doc_counts': {'table': 2, 'image_caption': 2, 'page_ocr': 1}}

[local] Modu Tech Q4 매출은?
  → Modu Tech의 2025년 4분기 매출은 188억원으로, 전년 동기 대비 35% 성장한 수치입니다. 이 기간 동안 영업이익은 24억원을 기록했으며, 실 솔루션 부문이 매출 증가에 크게 기여했습니다. 고객 수는 1,820명으로 45.6% 증가했으며, 월간 활성 사용자(MAU)는 132만으로 55.3% 증가했습니다. 고객 만족도(NPS)도 58로 상승하며,...


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryType(type='discussio...로 분류됩니다."), input_type=QueryType])
  return self.__pydantic_serializer__.to_python(



[adaptive] 회의에서 다룬 주요 결정은?
  → 회의에서 다룬 주요 결정은 2025년 4분기 실적 보고서와 관련된 다양한 AI 솔루션, 클라우드, 컨설팅 및 기타 제품의 매출과 영업이익을 분석하는 것이었습니다. 이 데이터는 표 형식으로 제공되어 각 제품의 성과를 전년 동기 대비 비교할 수 있도록 하였습니다. 또한, 2024년과 2025년의 고객 수, 월간 활성 사용자(MAU), 순추천지수(NPS), 이탈...

[global] 코퍼스 전반 주제는?
  → 코퍼스 전반의 주제는 기업의 성과 분석과 관련된 다양한 메트릭 및 데이터입니다. 커뮤니티 C0은 2025년 4분기 실적 보고서에 포함된 AI 솔루션, 클라우드, 컨설팅 제품의 매출과 영업이익을 다루고 있으며, C1은 고객 수, MAU, NPS, 이탈률 등의 주요 메트릭 변화를 분석합니다. C2와 C3은 수익 증가 및 매출과 시장 평균 간의 관계를 시각적으로...

⚖️  Comparison: graphrag — Graph-RAG는 매출 추이에 대한 분석을 그래프와 표를 통해 시각적으로 제시하며, 각 사업부의 매출과 영업이익, 성장률을 상세히 설명합니다. 또한, AI 솔루션 부문과 클라우드 부문의 성장 요인에 대한 분석을 포함하여, 매출 증가 추세와 향후 전략에 대한 통찰을 제공합니다. 반면, baseline은 매출 수치와 성장률을 간단히 나열하는 데 그쳐, 분석의 깊이가 부족합니다. 따라서 graphrag가 더 충실하고 풍부한 정보를 제공합니다.


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Comparison(better='graphr...를 제공합니다.'), input_type=Comparison])
  return self.__pydantic_serializer__.to_python(
